本次作业包含game.py、strategy.py，assignment4.ipynb三个文件。

game.py中实现了收益矩阵Payoff类和玩家基类Player类

strategy.py中实现了课上提到的四种策略（AlwaysCooperatePlayer，AlwaysDefectPlayer，DowningPlayer，RandomPlayer）

请在assignment4.ipynb中完成作业，并基于结尾部分的代码进行验证。

In [1]:
import random
import pandas as pd
from game import Payoff
from strategy import *

# 第一题（60分）
请实现以下几种策略：
1. TitForTatPlayer(5分)
2. TitForTwoTatsPlayer(5分)
3. DavisPlayer(5分)
4. GrudgerPlayer（5分）
5. ShubikPlayer（20分）
6. FeldPlayer（20分）

### 1. TitForTatPlayer
以牙还牙，对手上次选择什么，我就选择什么。在第一轮选择合作

In [2]:
#class TitForTatPlayer(Player):
class TitForTatPlayer(Player):
    def move(self,opponent: Player) -> str:
        if not self.history:
            return 'C'
        return opponent.history[-1]

### 2.TitForTwoTatsPlayer
前两轮选择合作，之后若对手连续两次背叛，则选择背叛，否则选择合作。

In [3]:
#class TitForTwoTatsPlayer(Player):
class TitForTwoTatsPlayer(Player):
    def move(self,opponent: Player) -> str:
        if len(self.history) <= 1:
            return 'C'
        if opponent.history[-1] == 'D' and opponent.history[-2] == 'D':
                return 'D'
        return 'C'

### 3.GrudgerPlayer
若对手从未背叛过，就一直合作；一旦对手背叛一次，就一直选择背叛。

In [8]:
#class GrudgerPlayer(Player):
class GrudgerPlayer(Player):
     def move(self,opponent: Player) -> str:
         if not self.history:
             return 'C'
         # for i in opponent.history:
         #     if i == 'D':
         #      return 'D'
         # return 'C'
         if 'D' in opponent.history:
             return 'D'
         return 'C'

### 4.DavisPlayer
前10轮一定合作，之后如果发现对方背叛过，则一直背叛，否则一直合作。

In [13]:
#class DavisPlayer(Player):
class DavisPlayer(Player):
    def move(self,opponent: Player) -> str:
        if len(self.history) <= 9:
            return 'C'
        #for i in opponent.history:
        #   if i == 'D':
        #       return 'D'
        #return 'C'
        if 'D' in opponent.history:
            return 'D'
        return 'C'

### 5. ShubikPlayer
1. 开局示好，第一回合选择合作
2. 以直报怨，如果对方在上一回合背叛（出“D”），而自己上一回合是合作的（出“C”），策略会立刻触发报复。
3. 逐步升级，每次触发报复时：
    
    立刻反击：当前回合直接背叛（D）。

    延长惩罚：记录这次背叛，并让未来的惩罚时间比上一次多1回合。

    例如：第一次被背叛时，惩罚1回合；第二次被背叛时，惩罚2回合，依此类推。

4. 有限惩罚后和解，在设定的惩罚回合数结束后，策略会自动恢复合作。比如：如果当前需要惩罚2回合，那么连续出两次“D”后，第三回合主动变回“C”。

In [14]:
### class ShubikPlayer(Player):
class ShubikPlayer(Player):
    def __init__(self, name: str):
        super().__init__(name)
        self.punishment_count = 0
        self.punishment_remaining_count = 0
        
    def move(self,opponent: Player) -> str:
        
        if not self.history:#开局示好
            return 'C'

        if self.punishment_remaining_count > 0:
            self.punishment_remaining_count -= 1
            return 'D'
        
        if opponent.history[-1] == 'D' and self.history[-1] == 'C':#以直报怨 触发报复
            self.punishment_count += 1
            self.punishment_remaining_count = self.punishment_count
            return 'D'
        return 'C'

### 6.费尔德策略FeldPlayer

费尔德策略是一种**合作概率随时间衰减的随机策略**：
1. 开局阶段倾向于合作 (`C`)。
2. 若对手上一回合背叛 (`D`)，则本回合必然背叛。
3. 随着回合数增加，合作概率逐渐降低至预设阈值。在对手上一轮合作的情况下，按当前合作概率随机选择合作或背叛。

#### 合作概率计算
定义当前回合的合作概率为：
$$
p_{\text{coop}} = \max\left( p_{\text{start}} + \left( \frac{p_{\text{end}} - p_{\text{start}}}{T_{\text{decay}}} \right) \cdot t, \ p_{\text{end}} \right)
$$

- **符号说明**：
  - $p_{\text{start}}$: 初始合作概率 (`start_coop_prob`)，设为1.0
  - $p_{\text{end}}$: 最终合作概率 (`end_coop_prob`)，设为0.5
  - $T_{\text{decay}}$: 合作概率衰减周期 (`rounds_of_decay`)，设为200
  - $t$: 当前已进行回合数 (`len(self.history)`)


In [15]:
#class FeldPlayer(Player):
class FeldPlayer(Player):

    def __init__(self, name: str):
        super().__init__(name)
        self.coop_prob = 0.0
        self.start_coop_prob = 1.0
        self.end_coop_prob = 0.5
        self.rounds_of_decay = 200
        
    def move(self,opponent: Player) -> str:
        if not self.history:
            return 'C'
        
        if opponent.history[-1] == 'D':
            return 'D'

        if opponent.history[-1] == 'C':
            self.coop_prob = max(self.start_coop_prob + (self.end_coop_prob - self.start_coop_prob) / self.rounds_of_decay * len(self.history)
                                 ,self.end_coop_prob)
            if random.random() < self.coop_prob:
                return 'C'
            return'D'

# 第二题（40分）
请补全Tournament类：

该类的属性包括：

| 属性名             | 类型                                                   | 含义与功能说明                                     |
|--------------------|--------------------------------------------------------|----------------------------------------------------|
| `players`          | `List[Player]`                                         | 所有参赛选手（策略）列表                          |
| `payoff`           | `Payoff`                                               | 收益矩阵，决定行动得分规则                        |
| `rounds`           | `int`                                                  | 每场比赛的轮数                                     |
| `_results`         | `Dict[str, int]`  | 存储每个玩家的累计总得分（按名称索引），即`Dict[玩家名, 累计得分]`，得分初始为0 |
| `_match_results`   | `Dict[Tuple[str, str], int]`| 每一组玩家对战中，player1 的得分，即`Dict[（player1名称, player2名称）, player1得分]` ，得分初始为0|
| `_round_history`   | `Dict[Tuple[str, str], List[Tuple[str, str]]]`| 存储每一组玩家在所有回合中的行为（如：`('C','D')`），即`Dict[（player1名称,player2名称）, List[（player1行动, player2行动），（player1行动, player2行动）......]`，list初始为空     |


该类的方法包括：
1. `_match(p1, p2)`：模拟两个玩家p1,p2之间的多轮对战；每轮获取双方动作，更新得分，并记录每一轮行为

    首先初始化双方得分和行为记录。
    然后，循环执行指定轮数的对战。
    在每一轮中，调用p1的move方法获取p1本轮应对p2的动作，然后调用p2的move方法获取p2本轮应对p1的动作。基于收益矩阵获取本轮双方的得分，加入总分中，并记录双方的行动到各自的行动记录中。
    最终，更新_round_history，返回双方各自的总得分。

2. `run()`：循环锦标赛的主控制流程；每两位玩家进行一次 _match()；累加得分并记录比赛结果到 _results 和 _match_results

    遍历每个玩家对（不包括自己与自己对战），对于每个玩家组合p1、p2，对玩家重置后，调用 _match 方法进行对战，将玩家p1的得分计入其总得分（更新_results），将p1与p2的对战结果记录在self._match_results中。   

3. ` __call__()`：让 Tournament 实例可以像函数一样调用；内部自动调用 run() 和 print_rankings()
4. `print_rankings()`：打印总排行榜（按平均得分降序）
5. `get_round_history(p1_name, p2_name)`：返回指定两位玩家对战的每一回合行为,用于复盘分析或可视化展示
6. `get_match_results()`：返回一个对战得分矩阵，行列为玩家名，元素为 player1 的得分，每行计算“平均得分”列供分析排名

以下代码已实现部分，请补充以下部分：
- 属性：`_results` ，`_match_results`，`_round_history`
- 方法：`_match`，`run`



In [44]:
class Tournament:

    def __init__(self, players: List[Player], payoff: Payoff, rounds: int = 5):
        self.payoff = payoff
        self.players = players
        self.rounds = rounds
        self._results: Dict[str,int] = {player.name: 0 for player in players}
        self._match_results: Dict[Tuple[str, str], int] = {}
        self._round_history: Dict[Tuple[str, str], List[Tuple[str, str]]] = {}

    def _match(self, p1: Player, p2: Player) -> Tuple[int, int]:

        p1.reset()
        p2.reset()

        score1 = 0
        score2 = 0
        round_actions = []

        for i in range(self.rounds):
            action1 = p1.move(p2)
            action2 = p2.move(p1)
            p1.record_moves(action1)
            p2.record_moves(action2)
            round_score = self.payoff.matrix[(action1,action2)]
            round_actions.append((action1,action2))
            score1 += round_score[0]
            score2 += round_score[1]

        self._round_history[(p1.name,p2.name)] = round_actions

        return (score1,score2)
        pass
    
    def run(self) -> None:
        for i in range(len(self.players)):
            for j in range(i+1,len(self.players)):
                p1 = self.players[i]
                p2 = self.players[j]

                score1,score2 = self._match(p1,p2)
                
                self._results[p1.name] += score1
                self._results[p2.name] += score2
                self._match_results[(p1.name,p2.name)] = score1

                score2_rev, score1_rev = self._match(p2, p1)
               
                self._results[p2.name] += score2_rev
                self._results[p1.name] += score1_rev
                self._match_results[(p2.name, p1.name)] = score2_rev
        
        pass
    
    def __repr__(self) -> str:
        return f"Tournament(players={len(self.players)}, rounds={self.rounds})"
    
    @property
    def rankings(self) -> List[Tuple[str, int]]:
        """返回根据得分降序排序后的排行榜"""
        return sorted(self._results.items(), key=lambda x: x[1], reverse=True)
    
    def print_rankings(self) -> None:
        print("=== Axelrod Tournament Results ===")
        for rank, (name, score) in enumerate(self.rankings, start=1):
            print(f"{rank}. {name:<24}: {score/len(self.players):.0f} points (on average)")

    def __call__(self) -> None:
        """使 Tournament 实例可直接调用以运行并输出"""
        self.run()
        self.print_rankings()

    def get_round_history(self, p1_name: str, p2_name: str) -> pd.DataFrame:
        '''
        返回指定玩家对战的回合记录，格式为pandas.DataFrame，包含2行N列，分别为player 1和player 2的行动。
        '''
        history = self._round_history.get((p1_name, p2_name), None)
        if history is None:
            raise ValueError(f"No round history found for {p1_name} vs {p2_name}")
        return pd.DataFrame.from_records(history, columns=[f"P1: {p1_name}", f"P2: {p2_name}"], 
            index=range(1, self.rounds + 1))


    def get_match_results(self) -> pd.DataFrame:
        '''
        以矩阵形式返回每对玩家的对战结果，以pandas.DataFrame格式返回。矩阵的行和列分别为玩家的名称。
        其中，第i行j列表示第i个玩家作为player 1且第j个玩家作为player 2时，player 1的得分和player 2的得分。
        该矩阵为对称矩阵，即第i行j列和第j行i列的值相同。
        '''
        player_names = [p for p, s in self.rankings]
        result_matrix = pd.DataFrame(index=player_names, columns=player_names, dtype=object)
        
        for (p1_name, p2_name), score1 in self._match_results.items():
            result_matrix.at[p1_name, p2_name] = score1
            
        result_matrix['Average Score'] = result_matrix.mean(axis=1).transform(lambda x: int(round(x)))
        
        return result_matrix
    
 

# main(用于验证)

In [45]:
payoff = Payoff(R=3, P=1, S=0, T=10)
payoff

Payoff Matrix:
                   player 2 
                   C       D
player 1  C     (3, 3)  (0, 10)
          D     (10, 0)  (1, 1)

In [46]:
players = [
    AlwaysCooperatePlayer("AlwaysCooperate"),
    AlwaysDefectPlayer("AlwaysDefect"),
    TitForTatPlayer("TitForTat"),
    TitForTwoTatsPlayer("TitForTwoTats"),
    DavisPlayer("Davis"),
    DowningPlayer("Downing", payoff),
    FeldPlayer("Feld"),
    GrudgerPlayer("Grudger"),
    RandomPlayer("Random"),
    ShubikPlayer("Shubik"),
]

In [47]:
random.seed(2025)
tournament = Tournament(players, payoff, rounds=200)
tournament()  # 直接调用 __call__ 方法

=== Axelrod Tournament Results ===
1. Downing                 : 1171 points (on average)
2. AlwaysDefect            : 931 points (on average)
3. Grudger                 : 885 points (on average)
4. TitForTat               : 883 points (on average)
5. Davis                   : 860 points (on average)
6. Feld                    : 850 points (on average)
7. Random                  : 829 points (on average)
8. TitForTwoTats           : 796 points (on average)
9. AlwaysCooperate         : 756 points (on average)
10. Shubik                  : 676 points (on average)


In [48]:
tmp_df = tournament.get_round_history('Feld', 'Shubik')
display(tmp_df)

,P1: Feld,P2: Shubik
1,C,C
2,C,D
3,D,D
4,D,D
5,D,D
...,...,...
196,D,D
197,D,D
198,D,D
199,D,D


In [49]:
tournament.get_match_results()

,Downing,AlwaysDefect,Grudger,TitForTat,Davis,Feld,Random,TitForTwoTats,AlwaysCooperate,Shubik,Average Score
Downing,NaN,200,206,602,288,485,1136,722,2000,233,652
AlwaysDefect,200,NaN,209,209,290,209,1046,218,2000,362,527
Grudger,226,199,NaN,600,600,266,1175,600,600,228,499
TitForTat,602,199,600,NaN,600,289,689,600,600,257,493
Davis,208,190,600,600,NaN,271,1051,600,600,229,483
Feld,959,199,244,273,262,NaN,804,297,922,226,465
Random,96,117,120,695,196,591,NaN,965,1195,138,457
TitForTwoTats,342,198,600,600,600,352,499,NaN,600,235,447
AlwaysCooperate,0,0,600,600,600,471,285,600,NaN,600,417
Shubik,223,191,208,251,280,235,1198,288,600,NaN,386
